# Домашняя работа №1.2 (Основы ML)

1) Взять реализацию KNN из семинара №3. Добавить туда возможность поиска ближайших соседей не по евклидову расстоянию, а по манхеттанскому и по косинусному. - **3 балла**
2) Взять реализацию Линейной регрессии из семинара №4 и добавить туда возможность использовать в качестве loss функции Huber Loss (https://en.wikipedia.org/wiki/Huber_loss)
Необходимо реализовать новый подсчет градиента, и добавить в класс возможность выбора Loss функции (δ -гиперпараметр модели) - **4 балла**

## Требования:

1. Использовать собственные реализации, а не sklearn
2. Сдать необходимо в ipynb формате
3. Показать, что все работает на данных из семинаров

## Решение

In [21]:
import pandas as ps
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from numpy.linalg import norm

### Данные из семинара

In [16]:
data = load_breast_cancer()
X, y = data['data'], data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=True)

### 1) Взять реализацию KNN из семинара №3. Добавить туда возможность поиска ближайших соседей не по евклидову расстоянию, а по манхеттанскому и по косинусному.

#### Код и пример из семинара

In [5]:
class MyKNN:
    def __init__(self, k = 3):
        self.k = k
    
    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
    
    def calculate_euc_distance(self, x, y):
        return np.sqrt(((x - y)**2).sum())
    
    def calculate_matrix(self, X_test):
        distances = np.zeros((X_test.shape[0], self.X_train.shape[0]))
        for i in range(X_test.shape[0]):
            for j in range(self.X_train.shape[0]):
                distances[i, j] = self.calculate_euc_distance(X_test[i], self.X_train[j])
        return distances
    
    def predict(self, X_test):
        matrix = self.calculate_matrix(X_test)
        matrix_idx = np.argsort(matrix, axis=1,)[:, :self.k]
        res_matrix = np.array([self.y_train[x] for x in matrix_idx]).mean(axis=1)
        return res_matrix

In [18]:
knn = MyKNN(3)
knn.fit(X_train, y_train)

In [20]:
my_preds = knn.predict(X_test)
metrics.f1_score(y_test, my_preds.astype(int))

0.8982035928143712

Точность KNN с евклидовой метрикой расстояния - `0.898`

#### Домашняя реализация

Добавим возможность использовать другуб метрику расстояния

$$E(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2} = \|\mathbf{x} - \mathbf{y}\|_2$$

$$M(\mathbf{x}, \mathbf{y}) = \sum_{i=1}^{n} |x_i - y_i| = \|\mathbf{x} - \mathbf{y}\|_1$$

$$C(\mathbf{x}, \mathbf{y}) = 1 - \frac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\| \cdot \|\mathbf{y}\|} = 1 - \frac{\sum_{i=1}^{n} x_i y_i}{\sqrt{\sum_{i=1}^{n} x_i^2} \cdot \sqrt{\sum_{i=1}^{n} y_i^2}}$$

In [23]:
class MyMetricsKNN:
    def __init__(self, k = 3):
        self.k = k
        self.metrics = {
            'euclidean': lambda x, y: np.sqrt(((x - y) ** 2).sum()),
            'manhattan': lambda x, y: np.abs(x - y).sum(),
            'cosine': lambda x, y: 1 - (np.dot(x, y) / (norm(x) * norm(y))) 
                                   if (norm(x) * norm(y)) != 0 else 1.0
        }
    
    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
    
    def calculate_euc_distance(self, x, y, metric):
        return self.metrics[metric.lower()](x, y)
    
    def calculate_matrix(self, X_test, metric):
        distances = np.zeros((X_test.shape[0], self.X_train.shape[0]))
        for i in range(X_test.shape[0]):
            for j in range(self.X_train.shape[0]):
                distances[i, j] = self.calculate_euc_distance(X_test[i], self.X_train[j], metric)
        return distances
    
    def predict(self, X_test, metric):
        matrix = self.calculate_matrix(X_test, metric)
        matrix_idx = np.argsort(matrix, axis=1,)[:, :self.k]
        res_matrix = np.array([self.y_train[x] for x in matrix_idx]).mean(axis=1)
        return res_matrix

In [24]:
knn = MyMetricsKNN(3)
knn.fit(X_train, y_train)

In [26]:
my_preds = knn.predict(X_test, 'manhattan')
metrics.f1_score(y_test, my_preds.astype(int))

0.9090909090909091

In [27]:
my_preds = knn.predict(X_test, 'cosine')
metrics.f1_score(y_test, my_preds.astype(int))

0.8711656441717791

Качество классификации выше по манхеттанскому растоянию, но ниже по косинусному чем по евкулидовому)